In [1]:
import sys
sys.path.append("../")
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.rc('text', usetex=True)
matplotlib.rcParams['text.latex.preamble'] = r"\usepackage{amsmath,amssymb}"
plt.style.use('plots.mplstyle')
import warnings
from py_code import XGBoost_IC_test_model
warnings.filterwarnings('ignore')
from matplotlib.patches import Patch
from pathlib import Path

# Evaluate Model Performance on single station

This code reads test events on which the model has been applied and makes plots based on output probabilities. The plots refer to proton events and protons SMONLY events to garantee a significant muon component. The plots shown are : Inverse cumulative distribution function (1 - cdf), probabity distribution function (pdf) and feature importance values of the model. Here all individual events are read and stations probabilities are put in a single list to show single stations performance. 

- Specify the directory in which the txt with the probabilities have been stored.
- Please save both in .png and .pdf format using a (7,6) or (16,9) aspect ratio.
- **Make sure to save the plots in the pictures/ directory and LABEL THE PLOTS DESCRIPTEVELY AND UNIQUELY**

In [2]:
# Specify model and label as well as feature directory FOR FEATURES IMPORTANCE PLOTS
label = "100160TeV_025deg_4FF_180R175h_Black_13petrain_13petest_500ns"
feat_imp_save_dir = "../models_ML/HAWCSim_array/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/"

# Specify proton txt directory
PROTONS_txt_dir = Path("../output_files_ML/HAWCSim_array/test_protons/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/")
SMONLY_txt_dir = Path("../output_files_ML/HAWCSim_array/test_protons_SMONLY/100160TeV_025deg_4FF_180R175h_Black/13petrain_13petest_500ns/")

In [3]:
# Get all stations for protons

y_test = []
p_test = []

for txt_file in PROTONS_txt_dir.glob("*.txt"):
    name = txt_file.stem
    txt_file_name = str(txt_file)
    rc, pr, y_te, p_te = np.loadtxt(txt_file_name).T
    p_test.append(p_te)
    y_test.append(y_te)

p_test = np.concatenate(p_test)
y_test = np.concatenate(y_test)

ValueError: need at least one array to concatenate

In [ ]:
# Get all stations for protons SMONLY

y_test_sm = []
p_test_sm = []

for txt_file in SMONLY_txt_dir.glob("*.txt"):
    name = txt_file.stem
    txt_file_name = str(txt_file)
    rc, pr, y_te, p_te = np.loadtxt(txt_file_name).T
    p_test_sm.append(p_te)
    y_test_sm.append(y_te)

p_test_sm = np.concatenate(p_test_sm)
y_test_sm = np.concatenate(y_test_sm)

In [ ]:
# Call the function in test .py to make the inverse cumulative

bins = 30

x_s, icdf_s = XGBoost_IC_test_model.binned_inv_cdf(p_test[y_test == 1], bins=bins)
x_b, icdf_b = XGBoost_IC_test_model.binned_inv_cdf(p_test[y_test == 0], bins=bins)
x_SM, icdf_SM = XGBoost_IC_test_model.binned_inv_cdf(p_test_sm[y_test_sm == 1], bins=bins)


u = np.linspace(0, 1, len(icdf_s), endpoint=True)
u_SM = u = np.linspace(0, 1, len(icdf_SM), endpoint=True)

du = u[1] - u[0]
du_SM = u_SM[1] - u_SM[0]

pdf_s = du / np.diff(icdf_s, append=icdf_s[-1])
pdf_b = du / np.diff(icdf_b, append=icdf_b[-1])

pdf_s = np.clip(pdf_s, 0, np.inf)
pdf_b = np.clip(pdf_b, 0, np.inf)

plt.figure(figsize=(7, 6))

plt.step(u, icdf_s, where="post", lw=1.5, color="red", label=r"stations with $\mu^{\pm}$")
plt.step(u, icdf_b, where="post", lw=1.5, color="black", label=r"stations without $\mu^{\pm}$")
plt.step(u_SM, icdf_SM, where="post", lw=1.5, color="BLUE", label=r"stations with single $\mu^{\pm}$")


plt.xlabel("$P_{\mu}^{i}$", fontsize = 24)
plt.ylabel("Inverse cdf", fontsize = 24)
plt.xticks([0, 0.2, 0.4, 0.6, 0.8, 1], fontsize = 24)
plt.yticks([0, 0.2, 0.4, 0.6, 0.8, 1], fontsize = 24)
plt.xlim(-0.05, 1.05)

plt.legend(frameon=True, fontsize=13, loc=(0.20,0.65), facecolor="white", framealpha=1, edgecolor = "black")
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("../pictures/100160TeV_025deg_3FF_180R175h_Black/13peSM_13peBKG_13peTEST_d0600_500ns/invcum_100160TeV_25z_13pe_500ns_0600d.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# Plot the probabilities 

bins = 30
plt.figure(figsize=(7, 6))

plt.hist(p_test[y_test == 1],bins=bins, density=True,histtype="stepfilled", color="red", alpha=0.1)
plt.hist(p_test[y_test == 1],bins=bins, density=True,histtype="step", color="red", lw=1.5)
plt.hist(p_test[y_test == 0],bins=bins, density=True,histtype="stepfilled", color="black", alpha=0.1)
plt.hist(p_test[y_test == 0],bins=bins, density=True,histtype="step", color="black", lw=1.5)

print("Number of true muons stations = ", len(y_test[y_test == 1]))
print("Number of non-muons stations = ", len(y_test[y_test == 0]))
 
handles = [
    Patch(facecolor=(1,0,0,0.1), edgecolor=(1,0,0,1), linewidth=1, label=r"stations with $\mu^{\pm}$"),
    Patch(facecolor=(0,0,0,0.1), edgecolor=(0,0,0,1), linewidth=1, label=r"stations without $\mu^{\pm}$"),]

plt.xlabel("Probability", fontsize = 24)
plt.ylabel("Normalized Counts [A.U.]", fontsize = 24)
plt.xticks([0, 0.2, 0.4, 0.6, 0.8, 1], fontsize = 24)
plt.yticks(fontsize = 24)

plt.legend(handles = handles, frameon=True, fontsize=18, facecolor="white", edgecolor = "black", framealpha=1, loc=(0.43,0.75))
plt.grid(True, alpha=0.2)
plt.tight_layout()
plt.savefig("../pictures/100160TeV_025deg_3FF_180R175h_Black/13peSM_13peBKG_13peTEST_d0600_500ns/probdist_100160TeV_25z_13pe_500ns_0600d.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def load_feat(path):
    fts, vals = [], []
    with open(path) as f:
        for l in f:
            if l.strip():
                n, v = l.strip().split()
                fts.append(n)
                vals.append(float(v))
    return np.array(fts), np.array(vals)


imp_feat, imp_vals = load_feat(feat_imp_save_path + "xgb_feature_importance_" + label[0] + ".txt")


idx = np.argsort(imp_vals)

plt.figure(figsize=(16, 9))

plt.bar(imp_feat[idx], imp_vals[idx], color="black", alpha=0.1)
plt.bar(imp_feat[idx], imp_vals[idx], facecolor='none', edgecolor='black', linewidth=1.5)

plt.ylabel("XGB Gain", fontsize=30)

plt.tick_params(axis='x', which='both', bottom=False, top=False, left=False, right=False)

plt.xticks(rotation = 90, fontsize = 24)
plt.yticks(fontsize = 24)

plt.yscale("log")
plt.gca().invert_xaxis()  

plt.grid(True, axis='y', alpha=0.2)

plt.savefig("../pictures/100160TeV_025deg_3FF_180R175h_Black/13peSM_13peBKG_13peTEST_d0600_500ns/feat_importances_100160TeV_25z_13pe_500ns_0600d.pdf", dpi=300, bbox_inches="tight")

plt.show()